# Comparing Ground Motion Models (GMMs) with eGSIM: **Model-to-Model**

This notebook demonstates how to use data downloaded from eGSIM to create custom visualisations to compare different ground motion models and to explore different types of comparisons than those available via the graphical user interface.

The aims of the notebook are as follows:

1) Demonstrate how to open and interact with the data generated when downloading a high-density binary (hdf5) file for a model-to-model comparison undertaken with eGSIM's online graphical interface

2) Demonstrate how to use the eGSIM API to make a model-to-model comparison.

3) Provide an illustration of how exploit the API programmatically in order to compare different aspects of the GMM in a customised manner.

## Access and Custom Plot Downloaded HDF5 Data

Initially we need only a small set of scientific python tools imported.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

An example downloaded file from the eGSIM graphical user interface comprising a set of four GMMs (BindiEtAl2014Rjb, CauzziEtAl2014, ChiouYoungs2014, KothaEtAl2020ESHM20) and multiple IMTs for a set of magnitudes and distances has been provided.

In [ ]:
prediction_data = pd.read_hdf("../data/downloads/egsim-predictions.hdf")
prediction_data

<b>Pay attention ... the median ground motions are given in terms of $\ln\left( IMT \right)$, so you need to take the exponent of this value to plot the ground motions themselves.<b>

Now to create a simple trellis plot comparing the median and standard deviations of the four GMMs in terms of attenuation with distance for three magnitudes (5, 6, and 7) and three IMTs (PGA, SA(0.2), SA(1.0)).

In [ ]:
# Create a trellis plot of ground motion attenuation
fig, axs = plt.subplots(3, 3, figsize=(10,10), sharex=True, sharey=True)
# Group the data according to the magnitude
mag_grps = prediction_data.groupby(("input rupture_parameter mag"))

# Set the colour and pretty label name for each GMM
gmm_colours = {
    "BindiEtAl2014Rjb": ("tab:red", "Bindi et al. (2014)"),
    "ChiouYoungs2014": ("tab:blue", "Chiou & Youngs (2014)"),
    "CauzziEtAl2014": ("tab:green", "Cauzzi et al. (2015)"),
    "KothaEtAl2020ESHM20": ("gold", "Kotha et al. (2020)\n[ESHM20]")
}

# For each magnitude in list 5, 6, and 7
for i, mag in enumerate([5.0, 6.0, 7.0]):
    mag_grp = mag_grps.get_group(mag)
    # For each IMT from PGA, SA(0.2), SA(1.0)
    for j, imt in enumerate(["PGA", "SA(0.2)", "SA(1.0)"]):
        for gmm, (col, gmm_label) in gmm_colours.items():
            # Get the median and standard deviation
            median = mag_grp[f"{imt} median {gmm}"]
            stddev = mag_grp[f"{imt} stddev {gmm}"]
            rrup = mag_grp["input distance_measure rrup"]
            axs[j, i].fill_between(rrup, np.exp(median - stddev), np.exp(median + stddev),
                                   color=col, alpha=0.2)
            axs[j, i].semilogx(rrup, np.exp(median), "-", lw=2, color=col, label=gmm_label)
        # Make axes pretty
        axs[j, i].grid(which="both")
        axs[j, i].set_xscale("log")
        axs[j, i].set_yscale("log")
        axs[j, i].set_xlim(1.0, 1000.0)
        axs[j, i].set_ylim(1.0E-4, 5.)
        axs[j, i].set_xticks([1, 10, 100, 1000],
                             ["1", "10", "100", "1000"],
                             rotation=-30)

for i, (mag, imt) in enumerate(zip([5.0, 6.0, 7.0], ["PGA", "SA(0.2)", "SA(1.0)"])):
    axs[i, 0].set_ylabel(f"{imt} (g)", fontsize=12)
    axs[-1, i].set_xlabel("Rupture Distance (km)", fontsize=12)
    axs[0, i].set_title("Mw %.2f" % mag, fontsize=14)
    
fig.tight_layout(w_pad=0.1)
axs[-1, -1].legend(loc="lower left", fontsize=11)
# [Optionally] Save the figure
#plt.savefig("./example_model2model_GMMs.jpg", format="jpg", dpi=300, bbox_inches="tight")

## Simple Workflow using the eGSIM API

Here we will illustrate how to use the API to retreive the data to produce the same plot as that above ...


First, below is a function that constructs and executes the query using the configurations provided by the user, then returning the results as a dataframe

In [ ]:
# Standard library imports for web API usage
import io
import requests
from typing import Optional


def get_egsim_predictions(
        model: list[str],
        imt: list[str],
        magnitudes: list[float],
        distances: list[float],
        rupture_params: Optional[dict] = None,
        site_params: Optional[dict] = None,
        data_format="hdf",
) -> pd.DataFrame:
    """Retrieve the ground motion predictions for the selected set of ground motion
    models and intensity measure types. Each prediction will be the result of a given
    model, imt, and scenario, which is a configurable set of Rupture parameters and
    Site parameters.

    Args:
    - model: ground motion model(s) (OpenQuake class names)
    - imt: intensity measure type(s) (e.g. PGA, PGV, SA(0.1) etc.)
    - magnitudes: list of magnitudes configuring each Rupture
    - distances: list of distances configuring each Site
    - rupture_params: dict of shared Rupture parameters (magnitude excluded)
    - site_params: dict of shared Site parameters (distance excluded)
    - data_format: the requested data format. "hdf" (the default, recommended) or "csv".
      HDF is more performant and support more data types, but it requires pytables
      (`pip install tables`)

    Returns:

    A [pandas DataFrame](https://pandas.pydata.org/docs/user_guide/dsintro.html#dataframe)
    
    Each row denotes a scenario (i.e., a combination of a given Rupture and Site)
    labelled by a unique integer id, incremental and starting from 0 (*), and each column
    denotes:

    - a computed prediction if the first chunk of the column name is an intensity
      measure type (e.g. "PGA median BindiEtAl2014Rjb"): in this case, the second chunk
      is the metric type (e.g. "median") and the third the predicting model
      ("BindiEtAl2014Rjb")
    
    - a scenario input property if the first chunk of the column name is the text "input"
      (e.g., "input distance_measure rrup"): in this case, the second
      chunk is the configuration data type ("distance_measure", "intensity_measure",
      "rupture_parameter", "site_parameter" or "uncategorized") and the third is the
      configuration data name ("rrup")
    """
    base_url = "https://egsim.gfz.de/api/query/predictions"
    # request parameters (concatenate with site_config and rupture_config):
    parameters = {}
    if site_params:
        parameters |= site_params
    if rupture_params:
        parameters |= rupture_params

    # add remaining parameters:
    parameters |= {
        'model': model,
        'imt': imt,
        'format': data_format,
        'mag': magnitudes,
        'dist': distances
    }
    # POST request to eGSIM
    response = requests.post(base_url, json=parameters)

    # check HTTPErrors (status_code >= 400):
    if response.status_code >= 400 and response.text:
        raise requests.exceptions.HTTPError(response.text)
    else:
        response.raise_for_status()  # raises if status >= 400 (with a default message)

    # `response.content` is the computed data, as in-memory file (bytes sequence)
    # in CSV or HDF format. Read it into a pandas.DataFrame:
    if parameters['format'] == 'hdf':
        # `pd.read_hdf` works for HDF files on disk. Workaround:
        with pd.HDFStore(
                "data.h5",  # apparently unused for in-memory data
                mode="r",
                driver="H5FD_CORE",  # create in-memory file
                driver_core_backing_store=0,  # for safety, just in case
                driver_core_image=response.content) as store:
            dframe = store[list(store.keys())[0]]
    else:
        # use `pd.read_csv` with a BytesIO (file-like object) as input:
        dframe = pd.read_csv(io.BytesIO(response.content), header=[0, 1, 2], index_col=0)

    return dframe

To make a successful query you need the following information:

`model` = List of ground motion models (from the OpenQuake class names)

`imt` = List of the requested intensity measures (from the OpenQuake class names, e.g. `PGA`, `PGV`, `SA(0.2)`, `SA(1.0)`, etc.)

`magnitudes` = List of magnitudes (must be floats)

`distances` = List of _rupture distances_ (`rrup`) in km (must be floats)

`rupture_params` = Dictionary of additional rupture parameters

`site_params` = Dictionary of additional site parameters

In [ ]:
gmms = ['BindiEtAl2014Rjb', 'ChiouYoungs2014', 'CauzziEtAl2014', 'KothaEtAl2020ESHM20']
imts = ["PGA", "SA(0.2)", "SA(1.0)"]
magnitudes = [5.0, 6.0, 7.0]
distances = [1.0, 2.0, 5.0, 7.5, 10.0, 15.0, 20.0, 30.0, 40.0, 50.0, 
             60.0, 80.0, 100.0, 125.0, 150.0, 175.0, 200.0, 225.0, 250.0]

# Other rupture parameters
rupture_params = {
    "dip": 90.0,   # Dip angle in decimal degrees
    "ztor": 0.0,  # Top of rupture depth (km)
    "rake": 0.0,  # Rake angle of rupture (degrees) following Aki & Richards convention:
                  # -90 = normal, 0/180 = strike slip, 90 = reverse
}
# Other site parameters
site_params = {
    "vs30": 800.0,  # Vs30 in m/s
    "z1pt0": None,  # Depth to 1.0 km/s Vs layer (m)
    "z2pt5": None,  # Depth to 2.5 km/s Vs layer (in km)
    "vs30measured": True,  # Vs30 is a measured (True) or inferred (False) value
    "line-azimuth": 90.0,  # Azimuth with respect to rupture strike to place the target site(s)
}

# Get the eGSIM model predictions
gmvs = get_egsim_predictions(
    model=gmms, 
    imt=imts, 
    magnitudes=magnitudes,
    distances=distances,
    rupture_params=rupture_params,
    site_params=site_params,
)
gmvs

... repeat the plotting process, this time with `gmvs`

In [ ]:
# Group the data according to the magnitude
mag_grps = gmvs.groupby(("input rupture_parameter mag"))

# Create a trellis plot of ground motion attenuation
fig, axs = plt.subplots(3, 3, figsize=(10,10), sharex=True, sharey=True)

# Set the colour and pretty label name for each GMM
gmm_colours = {
    "BindiEtAl2014Rjb": ("tab:red", "Bindi et al. (2014)"),
    "ChiouYoungs2014": ("tab:blue", "Chiou & Youngs (2014)"),
    "CauzziEtAl2014": ("tab:green", "Cauzzi et al. (2015)"),
    "KothaEtAl2020ESHM20": ("gold", "Kotha et al. (2020)\n[ESHM20]")
}

# For each magnitude in list 5, 6, and 7
for i, mag in enumerate([5.0, 6.0, 7.0]):
    mag_grp = mag_grps.get_group(mag)
    # For each IMT from PGA, SA(0.2), SA(1.0)
    for j, imt in enumerate(["PGA", "SA(0.2)", "SA(1.0)"]):
        for gmm, (col, gmm_label) in gmm_colours.items():
            # Get the median and standard deviation
            median = mag_grp[f"{imt} median {gmm}"]
            stddev = mag_grp[f"{imt} stddev {gmm}"]
            rrup = mag_grp["input distance_measure rrup"]
            axs[j, i].fill_between(rrup, np.exp(median - stddev), np.exp(median + stddev),
                                   color=col, alpha=0.2)
            axs[j, i].semilogx(rrup, np.exp(median), "-", lw=2, color=col, label=gmm_label)
        # Make axes pretty
        axs[j, i].grid(which="both")
        axs[j, i].set_xscale("log")
        axs[j, i].set_yscale("log")
        axs[j, i].set_xlim(1.0, 1000.0)
        axs[j, i].set_ylim(1.0E-4, 5.)
        axs[j, i].set_xticks([1, 10, 100, 1000],
                             ["1", "10", "100", "1000"],
                             rotation=-30)

for i, (mag, imt) in enumerate(zip([5.0, 6.0, 7.0], ["PGA", "SA(0.2)", "SA(1.0)"])):
    axs[i, 0].set_ylabel(f"{imt} (g)", fontsize=12)
    axs[-1, i].set_xlabel("Rupture Distance (km)", fontsize=12)
    axs[0, i].set_title("Mw %.2f" % mag, fontsize=14)
    
fig.tight_layout(w_pad=0.1)
axs[-1, -1].legend(loc="lower left", fontsize=11)

# More Advanced Workflow using the eGSIM API 

So far we have shown how to use the API to produce a customised trellis plot that is still effectively reproducing a feature available through the graphical interface. But what if we want to look at a different aspect of the GMMs to compare?

Here we compare several subduction (interface) ground motion models and their respective site amplifications:

* Abrahamson et al. (2016) "BC Hydro" - Global subduction GMM developed for application in northwest US & Canada
* Montalva et al. (2017) - Calibration of the BC Hydro model to Chilean strong motion data
* Abrahamson et al. (2018) - Update to the BC Hydro model using NGA Subduction data (for application to Northwest US & Canada)
* Abrahamson & Gulerce (2020) - NGA Subduction GMM (global)
* Kuehn et al. (2020) - NGA Subduction GMM (global)
* Parker et al. (2020) - NGA Subduction GMM (global)

In [ ]:
# Select 5 candidate subduction GMMs
subduction_gmms = {
    # Openquake Class Name: (Pretty name, plotting color, plotting marker)
    "AbrahamsonEtAl2015SInter": ("Abrahamson et al. (2016)", "tab:blue", "o"),
    "MontalvaEtAl2017SInter": ("Montalva et al. (2017)", "tab:grey", "v"),
    "AbrahamsonEtAl2018SInter": ("Abrahamson et al. (2018)", "k", "s"),
    "AbrahamsonGulerce2020SInter": ("Abrahamson & Gulerce (2022)", "tab:red", "D"),
    "KuehnEtAl2020SInter": ("Kuehn et al (2022)", "tab:green", "*"),
    "ParkerEtAl2020SInter": ("Parker et al. (2022)", "tab:orange", "P"),
}

Before setting up the configuration it can be helpful to know which inputs are required for the selected GMMs and therefore which optional properties may influence the comparisons. This can be done using the eGSIM API query, which returns a json with relevant attributes of each model.

In [ ]:
# Determine the attributes required for each GMM
r = requests.get("https://egsim.gfz.de/api/query/models?name={:s}".format(",".join(list(subduction_gmms))))
for key, attr in r.json().items():
    print(key, list(attr["requires"]))

To compare the site amplification we will call the eGSIM API several times each with a different Vs30 in order to retrieve the spectra
for our range of distances and magnitudes (and rupture/site properties). Below are the query properties that will remain constant with each call.

In [ ]:
# PGA and response spectra
periods = [0.02, 0.03, 0.05, 0.075, 0.1, 0.12, 0.14, 0.16, 0.18, 0.2,
           0.22, 0.24, 0.26, 0.28, 0.3, 0.32, 0.34, 0.36, 0.38, 0.4,
           0.42, 0.44, 0.46, 0.48, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8,
           0.85, 0.9, 0.95, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
imts = ["PGA",] + [f"SA({per})" for per in periods] 

magnitudes = [7.0, 8.0, 9.0]
distances = [20.0, 60.0, 120.0]

# Other rupture parameters
rupture_params = {
    "aspect": 3.0,  # Rupture aspect ratio (length/width)
    "dip": 45.0,
    "ztor": 5.0,
    "rake": 90.0
}
# Other site parameters
site_params = {
    "vs30": None,
    "z1pt0": None,
    "z2pt5": None,
    "vs30measured": True,
    "line-azimuth": 90.0, "backarc": False
}

In [ ]:
subduction_gmvs = {}
# Get ground motion values for different vs30 values and store as separate dataframes in a dictionary
vs30s = [1000.0, 600.0, 400.0, 200.0]
for vs30 in vs30s:
    site_params["vs30"] = vs30
    subduction_gmvs["{:.0f}".format(vs30)] = get_egsim_predictions(
        model=list(subduction_gmms), 
        imt=imts, 
        magnitudes=magnitudes,
        distances=distances,
        rupture_params=rupture_params,
        site_params=site_params,
    )
# Show the dataframe for rock (1000 m/s)
subduction_gmvs["1000"]

Now plot the amplification in ground motion response spectra for each of the other Vs30 values with respect to rock for a Mw 8 earthquake at the three different distances (20 km, 60 km and 120 km)

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(12,12), sharex=True, sharey=True, )
mag = 8.0
# Get the periods (here PGA = SA(0.01))
xvals = np.hstack([np.array([0.01]), periods])
# Get the ground motions for rock (Vs30 1000 m/s)
gmv_rock_grps = subduction_gmvs["1000"].groupby(["input distance_measure rrup",
                                                 "input rupture_parameter mag"])
for i, rrup in enumerate(distances):
    rock_grp = gmv_rock_grps.get_group((rrup, mag))
    for j, vs30_label in enumerate(["600", "400", "200"]):
        gmv_grps = subduction_gmvs[vs30_label].groupby(["input distance_measure rrup",
                                                        "input rupture_parameter mag"])
        grp = gmv_grps.get_group((rrup, mag))
        
        # Get the amplification for each GMM
        for gmm, (gmm_label, gmm_color, gmm_marker) in subduction_gmms.items():
            gmm_cols = [f"{imt} median {gmm}" for imt in imts]
            ampl = np.exp(grp[gmm_cols].to_numpy()) / np.exp(rock_grp[gmm_cols].to_numpy())
            axs[i, j].semilogx(xvals, ampl[0, :], marker=gmm_marker, color=gmm_color, lw=2, label=gmm_label)
        axs[i, j].set_xlim(0.01, 10.0)
        axs[i, j].grid(which="both")
        axs[i, j].tick_params(labelsize=12)

for i in range(3):
    axs[0, i].set_title(f"Mw = {mag}, Rrup = {rrup} km", fontsize=14)
    axs[i, 0].set_ylabel("Amplification\n" + r"$V_{S30}$ %.0f / $V_{S30}$ 1000 m/s" % vs30s[i + 1], fontsize=14)
    axs[-1, i].set_xlabel("Period (s)", fontsize=14)
fig.tight_layout(pad=0.1)
axs[0, 0].legend(loc="upper left", fontsize=14)
